## Step 0 — Gradient Checker

In [39]:
from typing import Callable
import numpy as np

In [40]:
def numeric_gradient(
    scalarFunction: Callable[[np.ndarray], float],
    featureValues: np.ndarray,
    h: float = 1e-5
) -> np.ndarray:
    gradient = np.zeros_like(featureValues, dtype=float)

    for i in range(featureValues.size):
        mask = np.zeros_like(featureValues, dtype=float)
        mask.flat[i] = h
        gradient.flat[i] = (scalarFunction(featureValues + mask) - scalarFunction(featureValues - mask)) / (2 * h)
    return gradient

def stable_softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - np.max(x))
    return e / np.sum(e)

### Tests — `numeric_gradient`

In [41]:
def _check(name, got, want, atol=1e-6):
    ok = np.allclose(got, want, atol=atol)
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    if not ok:
        print("   got :", got)
        print("   want:", want)

# 1. f = sum(x^2)  ->  grad = 2x        (vector input, nonlinear)
x = np.array([3.0, -1.0, 0.5])
_check("sum(x^2) grad == 2x", numeric_gradient(lambda v: np.sum(v**2), x), 2 * x)

# 2. f = sum(c*x)  ->  grad = c         (linear -> constant gradient)
c = np.array([2.0, -3.0, 0.7])
_check("sum(c*x) grad == c", numeric_gradient(lambda v: np.sum(c * v), np.zeros(3)), c)

# 3. matrix input -> gradient keeps the matrix shape
W = np.arange(6, dtype=float).reshape(2, 3)
g = numeric_gradient(lambda M: np.sum(M**2), W)
_check("matrix grad == 2W", g, 2 * W)
_check("matrix grad keeps shape", np.array(g.shape), np.array(W.shape))

# 4. f = sum(sin x) -> grad = cos x     (check vs analytic nonlinear)
x = np.array([0.1, 0.7, -1.2, 2.0])
_check("sum(sin x) grad == cos x", numeric_gradient(lambda v: np.sum(np.sin(v)), x), np.cos(x))

[PASS] sum(x^2) grad == 2x
[PASS] sum(c*x) grad == c
[PASS] matrix grad == 2W
[PASS] matrix grad keeps shape
[PASS] sum(sin x) grad == cos x


### Tests — `stable_softmax`

In [42]:
# reuses _check from the numeric_gradient test cell above
x = np.array([2.0, 1.0, 0.1])
p = stable_softmax(x)

_check("probs sum to 1", p.sum(), 1.0)
_check("all in (0, 1)", np.all((p > 0) & (p < 1)), True)

# matches the naive definition on small, safe inputs
naive = np.exp(x) / np.sum(np.exp(x))
_check("matches naive softmax", p, naive)

# stability + shift-invariance: huge logits stay finite and give the same result
big = stable_softmax(x + 1000)
_check("finite on x + 1000", np.all(np.isfinite(big)), True)
_check("shift-invariant (== p)", big, p)

# monotonic: largest logit keeps the largest probability
_check("argmax preserved", np.argmax(p), np.argmax(x))

[PASS] probs sum to 1
[PASS] all in (0, 1)
[PASS] matches naive softmax
[PASS] finite on x + 1000
[PASS] shift-invariant (== p)
[PASS] argmax preserved
